In [1]:
import os
import time
from pymmcore_plus import CMMCorePlus
from PyQt5.QtWidgets import QApplication, QMainWindow, QSlider, QLineEdit, QPushButton
import sys

class MicroscopeControlGUI(QMainWindow):
    def __init__(self):
        super().__init__()
        self.initUI()

    def initUI(self):
        # Change working directory
        os.chdir('C:/Users/Cell Culture Scope/Documents/MATLAB')

        # Delete all timers (if any in Micro-Manager context)
        # Note: Python doesn't have a direct equivalent to MATLAB's timerfindall and delete
        # You'll need to implement your own logic here if necessary

        # Slider Initialization
        self.dialampslider = QSlider(self)
        self.dialampslider.setMinimum(1)
        self.dialampslider.setMaximum(24)
        self.dialampslider.setValue(2)
        steps_dia = [1/50, 24/50]  # Step values
        self.dialampslider.setSingleStep(int(steps_dia[0]*100))

        self.EMslider = QSlider(self)
        self.EMslider.setMinimum(25)
        self.EMslider.setMaximum(51)
        self.EMslider.setSingleStep(1)

        # Micro-Manager Initialization
        self.mmc = CMMCorePlus.instance()
        self.mmc.loadSystemConfiguration("C:\\MATLAB Microscope\\AmoghMMConfig.cfg")

        Devices = self.mmc.getLoadedDevices()
        Devices_list = [Devices[i] for i in range(len(Devices))]

        self.camera = self.mmc.getCameraDevice()
        self.DIAshutter = self.mmc.getShutterDevice()
        self.focus = self.mmc.getFocusDevice()
        self.stage = self.mmc.getXYStageDevice()
        self.PFS = self.mmc.getAutoFocusDevice()
        self.EPIshutter = 'TIEpiShutter'
        self.DIAlamp = 'TIDiaLamp'
        self.scope = 'TIScope'
        self.zoom = 'TINosePiece'
        self.filter = 'TIFilterBlock1'
        self.lightpath = 'TILightPath'
        self.PFS_offset = 'TIPFSOffset'
        self.core = 'Core'

        CamProperties = self.mmc.getDevicePropertyNames(self.camera)
        self.CamProperties_list = [CamProperties[i] for i in range(len(CamProperties))]

        ScopeProperties = self.mmc.getDevicePropertyNames(self.scope)
        self.ScopeProperties_list = [ScopeProperties[i] for i in range(len(ScopeProperties))]

        DIALampProperties = self.mmc.getDevicePropertyNames(self.DIAlamp)
        self.DIALampProperties_list = [DIALampProperties[i] for i in range(len(DIALampProperties))]

        ZoomProperties = self.mmc.getDevicePropertyNames(self.zoom)
        self.ZoomProperties_list = [ZoomProperties[i] for i in range(len(ZoomProperties))]

        self.zoom4x = '1-(Achromat) 4x NA 0.10 Dry'
        self.zoom10x = '2-(Achromat) 10x NA 0.25 Dry'
        self.zoom20x = '3-(Achromat) 20x NA 0.40 Dry'
        self.zoom40x = '4-S Plan Fluor 40x NA 0.60 Dry'
        self.zoom60x = '5-Plan Apo 60x NA 1.40 Oil'
        self.zoomempty = '6-Unknown'

        FilterProperties = self.mmc.getDevicePropertyNames(self.filter)
        self.FilterProperties_list = [FilterProperties[i] for i in range(len(FilterProperties))]

        self.filter_Cy5_1 = '1-Cy5'
        self.filter_Cy3_2 = '2-Cy3'
        self.filter_3 = '3-TRITC'
        self.filter_FITC_4 = '4-FITC'
        self.filter_5 = '5-DAPI'
        self.filter_DIA_6 = '6-DIA'

        CoreProperties = self.mmc.getDevicePropertyNames(self.core)
        self.CoreProperties_list = [CoreProperties[i] for i in range(len(CoreProperties))]

        DIAShutterProperties = self.mmc.getDevicePropertyNames(self.DIAshutter)
        self.DIAShutterProperties_list = [DIAShutterProperties[i] for i in range(len(DIAShutterProperties))]

        EPIShutterProperties = self.mmc.getDevicePropertyNames(self.EPIshutter)
        self.EPIShutterProperties_list = [EPIShutterProperties[i] for i in range(len(EPIShutterProperties))]

        StageProperties = self.mmc.getDevicePropertyNames(self.stage)
        self.StageProperties_list = [StageProperties[i] for i in range(len(StageProperties))]

        LightPathProperties = self.mmc.getDevicePropertyNames(self.lightpath)
        self.LightPathProperties_list = [LightPathProperties[i] for i in range(len(LightPathProperties))]

        self.eyepath = '1-Eye100'
        self.camerapath = '3-Right100'

        # Initialization for the stage controls
        self.xyfastlight.setChecked(True)
        self.xymediumlight.setChecked(False)
        self.xyslowlight.setChecked(False)
        self.zfastlight.setChecked(True)
        self.zmediumlight.setChecked(False)
        self.zslowlight.setChecked(False)

        # Initialization
        self.mmc.setProperty(self.camera, 'CCDTemperatureSetPoint', -70)
        self.temperature70 = QPushButton(self)
        self.temperature70.setChecked(True)

        self.mmc.setProperty(self.camera, 'Exposure', 50)
        self.exposuretext = QLineEdit(self)
        self.exposuretext.setText('50')

        self.mmc.setProperty(self.camera, 'Binning', 2)
        self.mmc.setProperty(self.lightpath, 'Label', self.eyepath)
        self.eyepathlight = QPushButton(self)
        self.eyepathlight.setChecked(True)

        self.camerapathlight = QPushButton(self)
        self.camerapathlight.setChecked(True)

        self.mmc.setProperty(self.camera, 'Gain', 10)
        self.gaintext = QLineEdit(self)
        self.gaintext.setText('10')

        self.mmc.setProperty(self.camera, 'Pre-Amp-Gain', '5.1x')
        self.EMslider.setValue(51)
        self.EMtext = QLineEdit(self)
        self.EMtext.setText('5.1x')

        self.mmc.setProperty(self.camera, 'PixelType', '16bit')

        # Zoom Initialization
        zoom_ini = self.mmc.getProperty(self.zoom, 'Label')
        if zoom_ini == self.zoom4x:
            self.zoomlight4x.setChecked(True)
        elif zoom_ini == self.zoom10x:
            self.zoomlight10x.setChecked(True)
        elif zoom_ini == self.zoom20x:
            self.zoomlight20x.setChecked(True)
        elif zoom_ini == self.zoom40x:
            self.zoomlight40x.setChecked(True)
        elif zoom_ini == self.zoom60x:
            self.zoomlight60x.setChecked(True)

        # Light Path Initialization
        self.mmc.setProperty(self.lightpath, 'Label', self.eyepath)
        self.eyepathlight.setChecked(True)
        self.camerapathlight.setChecked(False)

        # Filter Initialization
        filter_ini = self.mmc.getProperty(self.filter, 'Label')
        if filter_ini == self.filter_Cy5_1:
            self.cy5filterlight.setChecked(True)
        elif filter_ini == self.filter_Cy3_2:
            self.cy3filterlight.setChecked(True)
        elif filter_ini == self.filter_DIA_3:
            self.DIAfilterlight.setChecked(True)

        # DIA Lamp Initialization
        self.mmc.setProperty(self.DIAlamp, 'ComputerControl', 'On')
        self.dialampmanuallight = QPushButton(self)
        self.dialampmanuallight.setChecked(False)
        self.dialampsoftwarelight = QPushButton(self)
        self.dialampsoftwarelight.setChecked(True)

        self.mmc.setProperty(self.DIAlamp, 'Intensity', 3)
        self.mmc.setProperty(self.DIAlamp, 'State', 1)
        self.dialamponlight = QPushButton(self)
        self.dialamponlight.setChecked(True)
        self.dialampofflight = QPushButton(self)
        self.dialampofflight.setChecked(False)
        self.dialampvalue = QLineEdit(self)
        self.dialampvalue.setText('2')

        # Shutter Initialization
        self.mmc.setProperty(self.DIAshutter, 'State', 0)
        self.mmc.setProperty(self.EPIshutter, 'State', 0)
        self.DIAshutterlight = QPushButton(self)
        self.DIAshutterlight.setChecked(False)
        self.EPIshutterlight = QPushButton(self)
        self.EPIshutterlight.setChecked(False)
        self.DIAshutterbutton = QPushButton(self)
        self.DIAshutterbutton.setChecked(False)
        self.EPIshutterbutton = QPushButton(self)
        self.EPIshutterbutton.setChecked(False)
        self.DIAshutterbutton.setStyleSheet('background-color: rgb(204, 204, 204);')
        self.EPIshutterbutton.setStyleSheet('background-color: rgb(204, 204);')

        # Finalizing GUI
        self.setWindowTitle('Microscope Control')
        self.show()

def main():
    app = QApplication(sys.argv)
    ex = MicroscopeControlGUI()
    sys.exit(app.exec_())

if __name__ == '__main__':
    main()


AttributeError: 'MicroscopeControlGUI' object has no attribute 'xyfastlight'